In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import shutil
import random
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
import json
import itertools

# =====================================================================
# 1. SETUP & PATH CONFIGURATION (KAGGLE PATHS)
# =====================================================================
SEED_BASE = 42
torch.manual_seed(SEED_BASE)
np.random.seed(SEED_BASE)
random.seed(SEED_BASE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_BASE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BACKUP_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
UNLABELED_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
OUTPUT_DIR = '/kaggle/working/Thesis_Outputs'
LABELED_DIR = '/kaggle/working/data/labeled_real'

os.makedirs(LABELED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(UNLABELED_DIR):
    raise FileNotFoundError(f"Could not locate image directory at {UNLABELED_DIR}")

# Load data splits
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

# Copy labeled images to working directory
for img_id in all_images:
    src = f'{UNLABELED_DIR}/{img_id}.jpg'
    dst = f'{LABELED_DIR}/{img_id}.jpg'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

print(f"Labeled images verified: {len(all_images)}")
print(f"Train class distribution: {train_df['label'].value_counts().to_dict()}")

# =====================================================================
# 2. ENCODER & CLASSIFIER (same as ablation script)
# =====================================================================
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

# =====================================================================
# 3. LOSS FUNCTIONS
# =====================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

# =====================================================================
# 4. AUGMENTATION TRANSFORMS
# =====================================================================
# Standard augmentation (used for baseline, focal, weights, oversampling)
normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Heavy MEL‑specific augmentation
mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# =====================================================================
# 5. DATASET WITH CONDITIONAL PARAMETERS
# =====================================================================
class AugmentedDataset(Dataset):
    def __init__(self, df, image_dir, mel_multiplier=5,
                 transform_normal=None, transform_mel=None):
        self.df = df
        self.image_dir = image_dir
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel

        mel_rows = df[df['label'] == 'MEL']
        other_rows = df[df['label'] != 'MEL']
        mel_expanded = pd.concat([mel_rows] * mel_multiplier, ignore_index=True)
        self.expanded_df = pd.concat([mel_expanded, other_rows], ignore_index=True)
        self.expanded_df = self.expanded_df.sample(frac=1, random_state=42).reset_index(drop=True)

    def __len__(self):
        return len(self.expanded_df)

    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        img = Image.open(f"{self.image_dir}/{row['image']}.jpg").convert('RGB')
        label = self.class_to_idx[row['label']]

        if row['label'] == 'MEL' and self.transform_mel is not None:
            img = self.transform_mel(img)
        elif self.transform_normal is not None:
            img = self.transform_normal(img)
        else:
            img = test_transform(img)  # fallback
        return img, label

# =====================================================================
# 6. TRAIN / EVAL HELPERS
# =====================================================================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, macro_f1, all_labels, all_preds

# =====================================================================
# 7. ABLATION CONDITIONS DEFINITION
# =====================================================================
# Each condition is a dict that overrides the default settings.
# Default: plain CrossEntropy, normal_transform, no oversampling (multiplier=1),
#          no MEL-specific augmentation, no class weights.
# We'll define the components explicitly.

conditions = [
    {
        'name': 'baseline',
        'loss_type': 'CE',                # plain CrossEntropy
        'use_weights': False,             # no class weights
        'aug_mel_heavy': False,           # use normal_transform for all
        'mel_multiplier': 1,              # no oversampling
        'focal_gamma': None,
    },
    {
        'name': 'focal_only',
        'loss_type': 'focal',
        'use_weights': False,
        'aug_mel_heavy': False,
        'mel_multiplier': 1,
        'focal_gamma': 1.5,
    },
    {
        'name': 'weights_only',
        'loss_type': 'CE',
        'use_weights': True,
        'aug_mel_heavy': False,
        'mel_multiplier': 1,
        'focal_gamma': None,
    },
    {
        'name': 'aug_only',
        'loss_type': 'CE',
        'use_weights': False,
        'aug_mel_heavy': True,            # apply mel_transform to MEL
        'mel_multiplier': 1,
        'focal_gamma': None,
    },
    {
        'name': 'over_only',
        'loss_type': 'CE',
        'use_weights': False,
        'aug_mel_heavy': False,
        'mel_multiplier': 5,
        'focal_gamma': None,
    },
    {
        'name': 'all_rebalance',
        'loss_type': 'focal',
        'use_weights': True,
        'aug_mel_heavy': True,
        'mel_multiplier': 5,
        'focal_gamma': 1.5,
    },
]

seeds = [42, 123, 2024]
all_results = {}

# Precompute class mapping for consistent indexing
classes = sorted(train_df['label'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
weight_dict = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}

# =====================================================================
# 8. MAIN ABLATION LOOP
# =====================================================================
for cond in conditions:
    cond_name = cond['name']
    print(f"\n{'#'*50}")
    print(f"CONDITION: {cond_name}")
    print(f"{'#'*50}")

    cond_results = {}
    for seed in seeds:
        print(f"\n--- Seed {seed} ---")
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        # ---- 7a. Build dataset according to condition ----
        # Determine transforms
        if cond['aug_mel_heavy']:
            tr_normal = normal_transform
            tr_mel = mel_transform
        else:
            tr_normal = normal_transform
            tr_mel = None   # MEL will also use normal_transform

        # Instantiate datasets with seed-specific shuffling
        train_ds = AugmentedDataset(train_df, LABELED_DIR,
                                    mel_multiplier=cond['mel_multiplier'],
                                    transform_normal=tr_normal,
                                    transform_mel=tr_mel)
        # Shuffle order depends on seed
        train_ds.expanded_df = train_ds.expanded_df.sample(frac=1, random_state=seed).reset_index(drop=True)

        val_ds = AugmentedDataset(val_df, LABELED_DIR,
                                  mel_multiplier=1,
                                  transform_normal=test_transform,
                                  transform_mel=None)
        test_ds = AugmentedDataset(test_df, LABELED_DIR,
                                   mel_multiplier=1,
                                   transform_normal=test_transform,
                                   transform_mel=None)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        val_loader   = DataLoader(val_ds, batch_size=16)
        test_loader  = DataLoader(test_ds, batch_size=16)

        # ---- 7b. Model ----
        encoder = SimpleEncoder()
        for param in encoder.parameters():
            param.requires_grad = True
        model = SSLClassifier(encoder, len(classes)).to(device)

        # ---- 7c. Loss and optimizer ----
        if cond['loss_type'] == 'focal':
            if cond['use_weights']:
                alpha_vals = torch.tensor([weight_dict[cls] for cls in classes], dtype=torch.float32).to(device)
            else:
                alpha_vals = None
            criterion = FocalLoss(alpha=alpha_vals, gamma=cond['focal_gamma'])
        else:  # CE
            if cond['use_weights']:
                weight_tensor = torch.tensor([weight_dict[cls] for cls in classes], dtype=torch.float32).to(device)
                criterion = nn.CrossEntropyLoss(weight=weight_tensor)
            else:
                criterion = nn.CrossEntropyLoss()

        optimizer = optim.Adam(model.parameters(), lr=0.0001)

        # ---- 7d. Training with early stopping (macro-F1) ----
        epochs = 30
        best_val_macro_f1 = -1.0
        patience = 7
        epochs_no_improve = 0
        best_epoch = -1
        ckpt_path = f'{OUTPUT_DIR}/ablation_{cond_name}_seed{seed}.pth'
        history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}

        for epoch in range(epochs):
            train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
            val_acc, val_macro_f1, _, _ = evaluate(model, val_loader)

            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
            history["val_macro_f1"].append(val_macro_f1)

            if (epoch + 1) % 3 == 0:
                print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val Acc={val_acc:.3f}, Val Macro-F1={val_macro_f1:.3f}")

            if val_macro_f1 > best_val_macro_f1:
                best_val_macro_f1 = val_macro_f1
                best_epoch = epoch + 1
                torch.save(model.state_dict(), ckpt_path)
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        print(f"Best checkpoint: epoch {best_epoch} (Val Macro-F1={best_val_macro_f1:.3f})")

        # ---- 7e. Test evaluation ----
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        test_acc, test_macro_f1, true_labels, pred_labels = evaluate(model, test_loader)
        report = classification_report(true_labels, pred_labels,
                                        target_names=classes, output_dict=True)
        mel_recall = report['MEL']['recall']

        print(f">>> Seed {seed}: Test Acc={test_acc:.3f}, Test Macro-F1={test_macro_f1:.3f}, MEL Recall={mel_recall:.1%} <<<")
        print(f"Predictions: {Counter(pred_labels)}")

        cond_results[seed] = {
            'test_accuracy': test_acc,
            'test_macro_f1': test_macro_f1,
            'mel_recall': mel_recall,
            'best_epoch': best_epoch,
            'best_val_macro_f1': best_val_macro_f1,
            'report': report,
            'history': history,
            'prediction_counts': dict(Counter(pred_labels)),
            'checkpoint_selection_metric': 'val_macro_f1',
        }

    all_results[cond_name] = cond_results

# =====================================================================
# 9. SUMMARY PER CONDITION (across 3 seeds)
# =====================================================================
print("\n" + "="*60)
print("FULL ABLATION SUMMARY (mean +/- std over 3 seeds)")
print("="*60)

summary = {}
for cond_name, res in all_results.items():
    accs = [res[s]['test_accuracy'] for s in seeds]
    f1s  = [res[s]['test_macro_f1'] for s in seeds]
    recs = [res[s]['mel_recall'] for s in seeds]
    mean_acc = np.mean(accs); std_acc = np.std(accs)
    mean_f1  = np.mean(f1s);  std_f1  = np.std(f1s)
    mean_rec = np.mean(recs); std_rec = np.std(recs)
    summary[cond_name] = {
        'mean_acc': mean_acc, 'std_acc': std_acc,
        'mean_f1': mean_f1, 'std_f1': std_f1,
        'mean_rec': mean_rec, 'std_rec': std_rec,
    }
    print(f"{cond_name:>15} : Acc={mean_acc:.3f}±{std_acc:.3f}, F1={mean_f1:.3f}±{std_f1:.3f}, MEL Recall={mean_rec:.1%}±{std_rec:.1%}")

# Save full results
with open(f'{OUTPUT_DIR}/rebalancing_ablation_full.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Save summary
with open(f'{OUTPUT_DIR}/rebalancing_ablation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")

Device: cuda
Labeled images verified: 691
Train class distribution: {'NV': 349, 'BKL': 99, 'MEL': 35}

##################################################
CONDITION: baseline
##################################################

--- Seed 42 ---
Epoch 3: Train=0.725, Val Acc=0.731, Val Macro-F1=0.310
Epoch 6: Train=0.725, Val Acc=0.731, Val Macro-F1=0.333
Early stopping at epoch 8
Best checkpoint: epoch 1 (Val Macro-F1=0.373)
>>> Seed 42: Test Acc=0.452, Test Macro-F1=0.358, MEL Recall=62.5% <<<
Predictions: Counter({np.int64(2): 43, np.int64(1): 40, np.int64(0): 21})

--- Seed 123 ---
Epoch 3: Train=0.723, Val Acc=0.731, Val Macro-F1=0.281
Epoch 6: Train=0.733, Val Acc=0.740, Val Macro-F1=0.377
Epoch 9: Train=0.745, Val Acc=0.721, Val Macro-F1=0.377
Epoch 12: Train=0.754, Val Acc=0.721, Val Macro-F1=0.376
Epoch 15: Train=0.745, Val Acc=0.750, Val Macro-F1=0.392
Epoch 18: Train=0.760, Val Acc=0.750, Val Macro-F1=0.392
Early stopping at epoch 18
Best checkpoint: epoch 11 (Val Macro-F1=0.400

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


>>> Seed 123: Test Acc=0.721, Test Macro-F1=0.308, MEL Recall=0.0% <<<
Predictions: Counter({np.int64(2): 101, np.int64(0): 3})

--- Seed 2024 ---
Epoch 3: Train=0.720, Val Acc=0.760, Val Macro-F1=0.386
Epoch 6: Train=0.716, Val Acc=0.760, Val Macro-F1=0.388
Epoch 9: Train=0.735, Val Acc=0.740, Val Macro-F1=0.377
Epoch 12: Train=0.737, Val Acc=0.740, Val Macro-F1=0.357
Early stopping at epoch 13
Best checkpoint: epoch 6 (Val Macro-F1=0.388)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


>>> Seed 2024: Test Acc=0.712, Test Macro-F1=0.277, MEL Recall=0.0% <<<
Predictions: Counter({np.int64(2): 103, np.int64(0): 1})

##################################################
CONDITION: focal_only
##################################################

--- Seed 42 ---
Epoch 3: Train=0.723, Val Acc=0.731, Val Macro-F1=0.333
Epoch 6: Train=0.725, Val Acc=0.740, Val Macro-F1=0.373
Epoch 9: Train=0.731, Val Acc=0.702, Val Macro-F1=0.435
Epoch 12: Train=0.737, Val Acc=0.712, Val Macro-F1=0.325
Epoch 15: Train=0.760, Val Acc=0.702, Val Macro-F1=0.466
Epoch 18: Train=0.737, Val Acc=0.702, Val Macro-F1=0.434
Epoch 21: Train=0.743, Val Acc=0.702, Val Macro-F1=0.380
Early stopping at epoch 22
Best checkpoint: epoch 15 (Val Macro-F1=0.466)


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


>>> Seed 42: Test Acc=0.654, Test Macro-F1=0.327, MEL Recall=25.0% <<<
Predictions: Counter({np.int64(2): 90, np.int64(1): 14})

--- Seed 123 ---
Epoch 3: Train=0.616, Val Acc=0.683, Val Macro-F1=0.346
Epoch 6: Train=0.615, Val Acc=0.683, Val Macro-F1=0.344
Epoch 9: Train=0.637, Val Acc=0.683, Val Macro-F1=0.431
Epoch 12: Train=0.632, Val Acc=0.692, Val Macro-F1=0.421
Epoch 15: Train=0.650, Val Acc=0.702, Val Macro-F1=0.443
Epoch 18: Train=0.660, Val Acc=0.702, Val Macro-F1=0.446
Epoch 21: Train=0.665, Val Acc=0.663, Val Macro-F1=0.420
Early stopping at epoch 21
Best checkpoint: epoch 14 (Val Macro-F1=0.467)
>>> Seed 123: Test Acc=0.663, Test Macro-F1=0.357, MEL Recall=12.5% <<<
Predictions: Counter({np.int64(2): 90, np.int64(1): 11, np.int64(0): 3})

--- Seed 2024 ---
Epoch 3: Train=0.631, Val Acc=0.712, Val Macro-F1=0.449
Epoch 6: Train=0.634, Val Acc=0.683, Val Macro-F1=0.414
Epoch 9: Train=0.642, Val Acc=0.673, Val Macro-F1=0.426
Early stopping at epoch 10
Best checkpoint: epoch 3 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


>>> Seed 2024: Test Acc=0.692, Test Macro-F1=0.388, MEL Recall=37.5% <<<
Predictions: Counter({np.int64(2): 95, np.int64(1): 9})

##################################################
CONDITION: all_rebalance
##################################################

--- Seed 42 ---
Epoch 3: Train=0.575, Val Acc=0.558, Val Macro-F1=0.402
Epoch 6: Train=0.647, Val Acc=0.615, Val Macro-F1=0.444
Epoch 9: Train=0.682, Val Acc=0.596, Val Macro-F1=0.432
Epoch 12: Train=0.661, Val Acc=0.596, Val Macro-F1=0.439
Epoch 15: Train=0.682, Val Acc=0.606, Val Macro-F1=0.440
Epoch 18: Train=0.676, Val Acc=0.654, Val Macro-F1=0.483
Epoch 21: Train=0.687, Val Acc=0.635, Val Macro-F1=0.458
Epoch 24: Train=0.697, Val Acc=0.635, Val Macro-F1=0.461
Early stopping at epoch 25
Best checkpoint: epoch 18 (Val Macro-F1=0.483)
>>> Seed 42: Test Acc=0.663, Test Macro-F1=0.533, MEL Recall=37.5% <<<
Predictions: Counter({np.int64(2): 60, np.int64(0): 32, np.int64(1): 12})

--- Seed 123 ---
Epoch 3: Train=0.578, Val Acc=0.577,

TypeError: keys must be str, int, float, bool or None, not int64